# Taller — Construye y ajusta tu propio clasificador de imágenes

Este taller comienza **después de la teoría de CNN y del notebook guiado de transfer learning**.

La meta no es memorizar la API de Keras. La meta es tomar decisiones sobre un problema real: construir datos propios, adaptar un modelo preentrenado y analizar qué tan bien funciona.

---

## Organización de la actividad

| Momento | Tiempo estimado | Meta |
|---|---:|---|
| **Teoría + notebook guiado** | Se realiza antes de este taller | Comprender CNN, transfer learning y fine-tuning |
| **Trabajo en clase** | ~60 min | Construir el dataset y entrenar el primer modelo con la base congelada |
| **Trabajo en casa** | ~1–2 h | Evaluar, hacer fine-tuning, comparar y analizar errores |

### Regla importante

Durante el trabajo en clase usaremos **train** y **validation**.  
El conjunto de **test** se reservará para el trabajo en casa, cuando hagamos la evaluación final.

---

## Entrega final

Debes entregar **este mismo notebook ejecutado**, incluyendo:

- definición del problema y estrategia de búsqueda;
- dataset propio;
- primer entrenamiento con transfer learning;
- evaluación en test;
- fine-tuning;
- comparación antes/después;
- matriz de confusión;
- análisis de al menos 5 errores;
- conclusiones.

## 0. Preparación del entorno

La búsqueda de imágenes se realizará con `ddgs`. Si estás en Google Colab, ejecuta la siguiente celda.

In [ ]:
!pip -q install -U ddgs

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

from pathlib import Path
import hashlib, io, random, shutil, time
import numpy as np
import matplotlib.pyplot as plt
import requests
from PIL import Image

import tensorflow as tf
import keras
from keras import layers
from ddgs import DDGS
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Keras:", keras.__version__)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

# PARTE A — TRABAJO EN CLASE (~60 min)

## Objetivo de esta parte

Antes de terminar la sesión debes haber conseguido:

1. definir un problema de clasificación de 3 clases;
2. descargar y revisar un dataset propio;
3. separar los datos en `train`, `validation` y `test`;
4. cargar EfficientNetB0 preentrenada en ImageNet;
5. congelar el backbone;
6. entrenar una nueva cabeza de clasificación;
7. observar `accuracy` y `val_accuracy`.

> **No evalúes todavía sobre `test`.** Ese conjunto queda reservado para el trabajo en casa.

### Distribución sugerida

- **10 min** — problema y términos de búsqueda;
- **20 min** — descarga, inspección y limpieza;
- **10 min** — separación y carga del dataset;
- **20 min** — transfer learning y primer entrenamiento.

# 1. Define tu problema — EN CLASE

Elige un problema de clasificación de **3 clases**. Las clases deben ser visualmente distinguibles.

Ejemplos: `espresso/cappuccino/latte`, `bus/motorcycle/bicycle`, tres especies de flores, frutas, tipos de calzado o instrumentos musicales.

Evita problemas donde la clase dependa de información que no aparece visualmente en la imagen.

## TODO 1 — Problema y estrategia de búsqueda

Define:

1. las tres clases;
2. dos o tres términos de búsqueda por clase;
3. una breve hipótesis: **¿qué características visuales esperas que use el modelo?**;
4. un posible sesgo o fuente de *data leakage* que podría aparecer al descargar imágenes desde un buscador.

In [ ]:
# TODO 1 — EN CLASE: define tus clases y búsquedas
# Problema: clasificar fotografías de tres instrumentos musicales.
# Los nombres de carpeta van sin tildes para evitar problemas de rutas.
SEARCHES = {
    "guitarra": [
        "acoustic guitar instrument photo",
        "classical guitar on stand",
        "guitarra acustica",
    ],
    "trompeta": [
        "trumpet brass instrument photo",
        "trumpet close up",
        "trompeta instrumento musical",
    ],
    "violin": [
        "violin instrument photo",
        "violin and bow close up",
        "violin instrumento musical",
    ],
}

SEARCHES

### Antes de descargar

Escribe debajo de esta celda una respuesta corta:

- ¿Qué características visuales esperas que diferencien tus clases?
- ¿Qué imágenes incorrectas esperas encontrar?
- ¿Qué sesgo o *data leakage* podría aparecer?

**Problema:** clasificar fotografías de tres instrumentos musicales: **guitarra**, **trompeta** y **violín**. Son objetos con formas muy distintas y fáciles de reconocer para una persona, pero guitarra y violín comparten material (madera), cuerdas y un cuerpo con forma de ocho, así que el problema no es trivial.

**Términos de búsqueda:** tres por clase, mezclando inglés (el buscador devuelve más fotos reales) y español. Se agregan palabras como *instrument photo*, *close up* o *on stand* para favorecer fotografías del objeto completo y evitar personas tocando, conciertos o dibujos.

**¿Qué características visuales esperas que diferencien las clases?** La trompeta debería ser la más fácil: metal dorado o plateado brillante, tubos curvados, pistones y la campana. Entre guitarra y violín espero que el modelo use la proporción entre el cuerpo y el mástil, la boca redonda de la guitarra frente a los agujeros en forma de *f* del violín, el clavijero en espiral (voluta) del violín, la presencia del arco y el tamaño relativo de las cuerdas y trastes.

**¿Qué imágenes incorrectas esperas encontrar?** Guitarras eléctricas o bajos dentro de guitarra; violas, chelos o contrabajos dentro de violín (se ven casi iguales en una foto sin escala); cornetas, fliscornos o trombones dentro de trompeta; además dibujos, íconos, logos de tiendas, partituras, juguetes, estuches cerrados y fotos de conciertos donde el instrumento ocupa una parte mínima de la imagen.

**Posible sesgo o *data leakage*:** muchas imágenes vendrán de catálogos de tiendas, con el instrumento sobre fondo blanco y marca de agua; si una clase tiene más fotos de catálogo que otra, el modelo podría aprender "fondo blanco" en lugar del instrumento. Las trompetas suelen fotografiarse sobre fondos oscuros y los violines junto a una persona o un atril, lo que crea correlaciones de fondo. También aparecerán **casi duplicados**: la misma foto de producto publicada en varios sitios con distinto tamaño o recorte. El hash SHA256 solo elimina copias exactas, así que una misma foto podría quedar en train y en test, inflando artificialmente el accuracy. Por último, texto en la imagen ("Yamaha Violin", nombre de la tienda) podría revelar la etiqueta.

# 2. Buscar y descargar imágenes — EN CLASE

La siguiente función busca imágenes y descarga los resultados. Intenta ignorar URLs fallidas, verificar las imágenes, evitar duplicados exactos con un hash y convertirlas a RGB.

> Se solicita contenido con licencia Creative Commons cuando el motor lo soporta. Eso no sustituye la revisión de licencias si el dataset se va a redistribuir.

In [ ]:
RAW_DIR = Path("data_raw")
RAW_DIR.mkdir(exist_ok=True)

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def download_image_dataset(searches, root=RAW_DIR, images_per_query=60, min_side=120, timeout=8):
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0"})

    # Incluye imágenes de ejecuciones anteriores para que volver a ejecutar
    # la descarga no agregue duplicados exactos al dataset.
    seen_hashes = {
        file_sha256(path)
        for path in Path(root).glob("*/*.jpg")
        if path.is_file()
    }

    for class_name, queries in searches.items():
        class_dir = root / class_name
        class_dir.mkdir(parents=True, exist_ok=True)
        saved = len(list(class_dir.glob("*.jpg")))
        print(f"\n=== {class_name} ===")

        for query in queries:
            print(f"Buscando: {query!r}")
            try:
                results = DDGS().images(
                    query=query,
                    safesearch="moderate",
                    max_results=images_per_query,
                    license_image="any",
                )
            except Exception as e:
                print("  Error en búsqueda:", e)
                continue

            for result in results:
                url = result.get("image")
                if not url:
                    continue
                try:
                    response = session.get(url, timeout=timeout)
                    response.raise_for_status()
                    data = response.content
                    digest = hashlib.sha256(data).hexdigest()
                    if digest in seen_hashes:
                        continue

                    with Image.open(io.BytesIO(data)) as img:
                        img = img.convert("RGB")
                        if min(img.size) < min_side:
                            continue
                        filename = class_dir / f"{saved:04d}.jpg"
                        img.save(filename, format="JPEG", quality=90)

                    seen_hashes.add(digest)
                    saved += 1
                except Exception:
                    pass

            print(f"  Total guardadas hasta ahora: {saved}")
            time.sleep(1)

    print("\nDescarga terminada.")

## TODO 2 — EN CLASE: construye y limpia tu dataset

Tu objetivo inicial es obtener aproximadamente **100–150 imágenes útiles por clase**.

1. Ejecuta la descarga.
2. Inspecciona muestras aleatorias.
3. Mejora los términos de búsqueda si una clase trae resultados pobres.
4. Elimina imágenes claramente incorrectas o problemáticas.
5. Reporta el número final de imágenes por clase y describe el error más frecuente que encontraste.

> Si la búsqueda trae datos malos, **no aumentes simplemente `max_results`**. Primero mejora los términos de búsqueda.

In [ ]:
# TODO 2 — EN CLASE: ejecuta la descarga con tus búsquedas
download_image_dataset(
    SEARCHES,
    images_per_query=60,
)

In [ ]:
def count_images(root):
    counts = {}
    for folder in sorted(Path(root).iterdir()):
        if folder.is_dir():
            counts[folder.name] = len(list(folder.glob("*.jpg")))
    return counts

count_images(RAW_DIR)

### Limpieza adicional: casi duplicados

El hash SHA256 solo detecta copias idénticas byte a byte. Para reducir el *data leakage* entre splits, esta celda calcula un **hash perceptual** sencillo (*average hash* de 8x8 en grises) y elimina imágenes que se ven prácticamente iguales aunque cambie el tamaño o la compresión. Se compara contra todas las clases, porque una misma foto en dos carpetas también sería un problema.

In [ ]:
def average_hash(path, size=8):
    with Image.open(path) as img:
        g = np.asarray(img.convert("L").resize((size, size), Image.LANCZOS), dtype=np.float32)
    return (g > g.mean()).flatten()

def remove_near_duplicates(root, max_distance=3):
    kept_hashes, kept_paths, removed = [], [], []
    for path in sorted(Path(root).glob("*/*.jpg")):
        try:
            h = average_hash(path)
        except Exception:
            continue
        if kept_hashes:
            dists = np.count_nonzero(np.array(kept_hashes) != h, axis=1)
            j = int(np.argmin(dists))
            if dists[j] <= max_distance:
                print(f"Casi duplicado: {path} ≈ {kept_paths[j]}")
                path.unlink()
                removed.append(path)
                continue
        kept_hashes.append(h)
        kept_paths.append(path)
    print(f"\nCasi duplicados eliminados: {len(removed)}")
    return removed

near_dups = remove_near_duplicates(RAW_DIR)
count_images(RAW_DIR)

## Validación técnica automática de las imágenes

Antes de inspeccionar el contenido del dataset, comprobaremos que **TensorFlow pueda decodificar todas las imágenes**.

Esto resuelve un problema distinto al de la inspección visual:

- **Validación técnica:** ¿el archivo puede ser leído correctamente por TensorFlow?
- **Validación semántica:** ¿la imagen realmente pertenece a la clase indicada?

Una imagen puede abrirse aparentemente bien en otras librerías y aun así fallar durante `model.fit()`. Por eso hacemos esta comprobación usando el mismo decoder que utilizará el pipeline de Keras.

In [ ]:
def remove_invalid_images(root_dir):
    root_dir = Path(root_dir)
    bad_files = []

    for path in root_dir.rglob("*.jpg"):
        try:
            raw = tf.io.read_file(str(path))

            img = tf.io.decode_image(
                raw,
                channels=3,
                expand_animations=False,
            )

            # Forzamos la ejecución para detectar errores de decodificación.
            _ = img.numpy()

        except Exception as e:
            print("Imagen inválida:", path)
            print(" ", str(e).splitlines()[0])
            bad_files.append(path)

    for path in bad_files:
        path.unlink(missing_ok=True)

    print(f"\nImágenes inválidas eliminadas: {len(bad_files)}")

    return bad_files


bad_files = remove_invalid_images(RAW_DIR)

### Verificación

Ejecuta nuevamente la función. El resultado esperado es:

```text
Imágenes inválidas eliminadas: 0
```

Si todavía aparece alguna imagen inválida, vuelve a ejecutar hasta obtener cero antes de continuar.

In [ ]:
bad_files = remove_invalid_images(RAW_DIR)

assert len(bad_files) == 0, (
    "Todavía quedan imágenes que TensorFlow no puede decodificar."
)

print("✓ Todas las imágenes pueden ser decodificadas por TensorFlow.")

# 3. Inspección y limpieza semántica — EN CLASE

La etapa anterior confirmó que los archivos son **técnicamente válidos**.

Ahora viene una tarea diferente: comprobar si los datos son **semánticamente correctos**.

Busca especialmente:

- imágenes que no correspondan a la clase;
- dibujos, logos o diagramas si tu objetivo son fotografías;
- imágenes con texto que pueda revelar artificialmente la etiqueta;
- duplicados o imágenes casi idénticas;
- fondos que estén correlacionados con una clase;
- diferencias sistemáticas de estilo o calidad entre clases.

> Que TensorFlow pueda abrir una imagen no significa que sea un buen dato de entrenamiento.

In [ ]:
def show_random_images(root, n_per_class=6):
    root = Path(root)
    class_dirs = [p for p in sorted(root.iterdir()) if p.is_dir()]
    fig, axes = plt.subplots(len(class_dirs), n_per_class, figsize=(3*n_per_class, 3*len(class_dirs)))
    if len(class_dirs) == 1:
        axes = np.expand_dims(axes, 0)
    for row, class_dir in enumerate(class_dirs):
        files = list(class_dir.glob("*.jpg"))
        chosen = random.sample(files, min(n_per_class, len(files)))
        for col in range(n_per_class):
            ax = axes[row, col]
            if col < len(chosen):
                img = Image.open(chosen[col])
                ax.imshow(img)
                ax.set_title(class_dir.name)
            ax.axis("off")
    plt.tight_layout()

show_random_images(RAW_DIR)

### Revisión completa por clase y eliminación manual

Las muestras aleatorias sirven para una primera impresión, pero para limpiar hay que mirar **todas** las imágenes. La siguiente función muestra una hoja de contactos con el nombre de cada archivo; después se anotan los archivos a eliminar en `TO_DELETE` y se borran en la celda siguiente.

In [ ]:
def contact_sheet(class_name, cols=10, start=0, n=100):
    files = sorted((RAW_DIR / class_name).glob("*.jpg"))[start:start + n]
    rows = int(np.ceil(len(files) / cols))
    plt.figure(figsize=(1.8 * cols, 2.0 * rows))
    for i, f in enumerate(files):
        ax = plt.subplot(rows, cols, i + 1)
        with Image.open(f) as img:
            ax.imshow(img)
        ax.set_title(f.name, fontsize=7)
        ax.axis("off")
    plt.suptitle(f"{class_name}: imágenes {start} a {start + len(files) - 1}")
    plt.tight_layout()
    plt.show()

for class_name in SEARCHES:
    total = len(list((RAW_DIR / class_name).glob("*.jpg")))
    for start in range(0, total, 100):
        contact_sheet(class_name, start=start)

In [ ]:
# Archivos a eliminar después de revisar las hojas de contacto.
# Ejemplo: "violin": ["0007.jpg", "0031.jpg"]
TO_DELETE = {
    "guitarra": [],
    "trompeta": [],
    "violin": [],
}

before = count_images(RAW_DIR)
for class_name, names in TO_DELETE.items():
    for name in names:
        (RAW_DIR / class_name / name).unlink(missing_ok=True)
after = count_images(RAW_DIR)

print("Antes :", before)
print("Después:", after)
print("Eliminadas por clase:", {k: before[k] - after[k] for k in before})

### Checkpoint del dataset

Antes de continuar, asegúrate de que:

- cada carpeta contiene imágenes realmente asociadas a su clase;
- no predominan logos, dibujos o texto que revele la etiqueta;
- no hay una diferencia de fondo/estilo demasiado obvia entre clases;
- el número de imágenes por clase es razonablemente balanceado.

Registra aquí el número de imágenes eliminadas y el problema de calidad más frecuente.

**Registro de limpieza:**

| Clase | Descargadas | Casi duplicados | Eliminadas a mano | Finales |
|---|---:|---:|---:|---:|
| guitarra | | | | |
| trompeta | | | | |
| violin | | | | |

*(Llenar con las salidas de `count_images`, `remove_near_duplicates` y la celda de `TO_DELETE`.)*

**Problema de calidad más frecuente:** se espera que sea la **mezcla con instrumentos parecidos**: guitarras eléctricas y bajos en guitarra, y violas o chelos en violín, seguida de imágenes de catálogo con fondo blanco y marca de agua. Si una clase quedó con muchas menos imágenes que las otras, conviene agregar un término de búsqueda más específico para esa clase antes de entrenar.

# 4. Separar train, validation y test — EN CLASE

> **Concepto clave — tres conjuntos, tres funciones**
>
> - **Train:** se utiliza para actualizar los pesos del modelo.
> - **Validation:** se utiliza durante el desarrollo para decidir épocas, `learning_rate`, número de capas a descongelar, etc.
> - **Test:** se reserva para medir el rendimiento final. No debe guiar las decisiones de entrenamiento.

Usaremos **70 % entrenamiento, 15 % validación y 15 % prueba**.

In [ ]:
SPLIT_DIR = Path("dataset")

def split_dataset(source_root, destination_root, train_ratio=0.70, val_ratio=0.15, seed=SEED):
    source_root = Path(source_root)
    destination_root = Path(destination_root)
    if destination_root.exists():
        shutil.rmtree(destination_root)
    rng = random.Random(seed)

    for class_dir in sorted(source_root.iterdir()):
        if not class_dir.is_dir():
            continue
        files = list(class_dir.glob("*.jpg"))
        rng.shuffle(files)
        n = len(files)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)
        splits = {
            "train": files[:n_train],
            "validation": files[n_train:n_train+n_val],
            "test": files[n_train+n_val:],
        }
        for split_name, split_files in splits.items():
            target = destination_root / split_name / class_dir.name
            target.mkdir(parents=True, exist_ok=True)
            for src in split_files:
                shutil.copy2(src, target / src.name)

split_dataset(RAW_DIR, SPLIT_DIR)
for split in ["train", "validation", "test"]:
    print(split, count_images(SPLIT_DIR / split))

### Comprobación conceptual

Explica brevemente: ¿por qué usar repetidamente el conjunto de prueba para escoger épocas, `learning_rate` o capas a descongelar termina contaminando nuestra estimación de desempeño?

**Respuesta:**

Porque cada vez que miramos el resultado en test y cambiamos algo (épocas, `learning_rate`, capas a descongelar) estamos **ajustando el modelo a ese conjunto en particular**, aunque nunca entrenemos directamente con él. Después de probar muchas configuraciones y quedarnos con la que mejor sale en test, el número reportado ya no es una estimación honesta: es el máximo de muchos intentos y aprovecha las particularidades (y la suerte) de esas pocas imágenes. Con un test pequeño, como el nuestro de unas 20 imágenes por clase, una sola imagen cambia el accuracy varios puntos, así que el sesgo puede ser grande. El test pasa a funcionar como otro conjunto de validación y perdemos la única medida de cómo se comportará el modelo con datos realmente nuevos. Por eso todas las decisiones se toman con validation y test se usa una sola vez al final.

# 5. Construir los datasets de Keras — EN CLASE

`image_dataset_from_directory()` construye un `tf.data.Dataset` usando el nombre de las carpetas como clases.

Observa que:

- `train` se mezcla (`shuffle=True`);
- `validation` y `test` mantienen un orden estable;
- todas las imágenes se redimensionan a `224×224` para EfficientNetB0.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = keras.utils.image_dataset_from_directory(
    SPLIT_DIR / "train", image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="int", shuffle=True, seed=SEED)
val_ds = keras.utils.image_dataset_from_directory(
    SPLIT_DIR / "validation", image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="int", shuffle=False)
test_ds = keras.utils.image_dataset_from_directory(
    SPLIT_DIR / "test", image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="int", shuffle=False)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)
print(class_names)

# 6. Primer modelo: transfer learning — EN CLASE

Usaremos **EfficientNetB0** preentrenada en ImageNet. Primero congelaremos completamente la red base y entrenaremos solo una nueva cabeza de clasificación.

> **Concepto clave — Data augmentation**
>
> `RandomFlip`, `RandomRotation` y `RandomZoom` generan variaciones plausibles durante el entrenamiento. No crean información nueva, pero ayudan a que el modelo dependa menos de detalles accidentales de las imágenes descargadas.

> **Concepto clave — GlobalAveragePooling2D**
>
> En lugar de aplanar todos los mapas de características con `Flatten`, `GlobalAveragePooling2D` resume cada canal en un solo valor. Así reducimos mucho el número de parámetros de la cabeza de clasificación.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
], name="augmentation")

base_model = keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet", input_shape=(*IMG_SIZE, 3))
base_model.trainable = False

inputs = keras.Input(shape=(*IMG_SIZE, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = keras.Model(inputs, outputs)
model.summary()

## TODO 3 — EN CLASE: entrena la cabeza de clasificación

La base EfficientNet está congelada. Tu trabajo es configurar y entrenar la nueva cabeza.

> **Concepto clave — EarlyStopping**
>
> `EarlyStopping` detiene el entrenamiento cuando `val_loss` deja de mejorar. Con `restore_best_weights=True`, Keras recupera los pesos de la mejor época observada en validación.

Antes de ejecutar, escribe una predicción corta:

- ¿esperas que `train_accuracy` suba rápidamente?
- ¿qué señal en las curvas indicaría overfitting?

Completa únicamente los espacios `TODO` del código siguiente.

**Predicción antes de entrenar:**

Sí espero que `train_accuracy` suba muy rápido, probablemente por encima de 90 % en las primeras dos o tres épocas. La base EfficientNetB0 ya extrae características muy buenas de ImageNet (que incluye clases como *acoustic guitar*, *violin* y *cornet*), y solo estamos entrenando una capa densa de $1280 \times 3 + 3 = 3843$ parámetros.

La señal de overfitting sería que `accuracy` siga subiendo mientras `val_accuracy` se estanca o baja, o, más claro aún, que `val_loss` empiece a subir mientras `loss` sigue bajando. Para eso está el `EarlyStopping` sobre `val_loss`. Como hay data augmentation y dropout durante el entrenamiento, es posible incluso que al inicio `val_accuracy` sea mayor que `accuracy`.

In [ ]:
# TODO 3 — EN CLASE: completa la configuración de entrenamiento

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stopping],
)

### Curvas de aprendizaje — EN CLASE si el tiempo lo permite

Grafica `accuracy`, `val_accuracy`, `loss` y `val_loss`.

Si el tiempo de clase termina antes, **puedes completar estas curvas en casa**.  
Lo obligatorio antes de salir es haber entrenado el primer modelo y tener sus métricas de entrenamiento/validación.

In [ ]:
# Completa las cuatro curvas usando el objeto history
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="train accuracy")
plt.plot(history.history["val_accuracy"], label="val accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

best = int(np.argmin(history.history["val_loss"]))
print(f"Épocas ejecutadas: {len(history.history['loss'])}")
print(f"Mejor época (mínimo val_loss): {best + 1}")
print(f"  train accuracy = {history.history['accuracy'][best]:.3f}")
print(f"  val accuracy   = {history.history['val_accuracy'][best]:.3f}")
print(f"  val loss       = {history.history['val_loss'][best]:.4f}")

# PUNTO DE CONTROL — FIN DEL TRABAJO EN CLASE

Antes de continuar, verifica que tu equipo tenga:

- [ ] 3 clases definidas;
- [ ] términos de búsqueda documentados;
- [ ] imágenes descargadas;
- [ ] dataset inspeccionado y parcialmente limpiado;
- [ ] separación `train / validation / test`;
- [ ] EfficientNetB0 con pesos de ImageNet;
- [ ] backbone congelado;
- [ ] nueva cabeza de clasificación;
- [ ] al menos un entrenamiento completado;
- [ ] `accuracy` y `val_accuracy` reportadas.

## Hasta aquí llega el trabajo de clase.

**No uses todavía el conjunto de test para tomar decisiones.**

---

# PARTE B — TRABAJO EN CASA

A partir de este punto debes completar el análisis de manera autónoma.

Ahora sí vas a:

1. evaluar el modelo en `test`;
2. estudiar la matriz de confusión y métricas por clase;
3. realizar fine-tuning;
4. comparar el modelo congelado contra el modelo ajustado;
5. analizar errores;
6. escribir las conclusiones.

# 7. Evaluación del primer modelo — EN CASA

Ahora sí utilizaremos el conjunto de prueba.

> **Concepto clave — No todo es accuracy**
>
> - **Precision:** de las imágenes que el modelo predijo como una clase, ¿cuántas eran correctas?
> - **Recall:** de las imágenes que realmente pertenecían a una clase, ¿cuántas encontró?
> - **F1:** combina precision y recall.
> - **Matriz de confusión:** muestra qué clases se están confundiendo entre sí.

Guarda esta evaluación: será nuestra línea base antes del fine-tuning.

In [ ]:
test_loss_before, test_acc_before = model.evaluate(test_ds, verbose=0)
print(f"Test accuracy antes del fine-tuning: {test_acc_before:.4f}")

In [ ]:
def get_predictions(model, dataset):
    y_true = np.concatenate([y.numpy() for _, y in dataset])
    probabilities = model.predict(dataset, verbose=0)
    y_pred = probabilities.argmax(axis=1)
    return y_true, y_pred, probabilities

y_true, y_pred, probabilities = get_predictions(model, test_ds)
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names).plot(cmap="Blues", xticks_rotation=45)
plt.title("Matriz de confusión — antes del fine-tuning")
plt.show()

## TODO 4 — EN CASA: interpreta la evaluación

Utilizando `classification_report` y la matriz de confusión, responde:

1. ¿Cuál clase obtuvo mejores resultados?
2. ¿Cuál fue la principal confusión?
3. ¿La accuracy global oculta algún problema por clase?
4. ¿Qué cambiarías primero en el **dataset** antes de cambiar el modelo?

**Respuesta** *(los valores entre corchetes se completan con la salida de `classification_report` y la matriz de confusión)*:

1. **Mejor clase:** se espera que sea **trompeta** (F1 = [ ]). Es la clase visualmente más distinta: metal brillante, tubos y campana, sin parecido con los dos instrumentos de cuerda.
2. **Principal confusión:** se espera entre **guitarra y violín** ([ ] violines predichos como guitarra y [ ] guitarras predichas como violín). Ambos son de madera, con cuerdas, mástil y cuerpo en forma de ocho; en fotos donde no se aprecia la escala ni detalles como la voluta o los agujeros en *f*, son fáciles de confundir.
3. **¿La accuracy global oculta problemas?** Con tres clases más o menos balanceadas la accuracy global ([ ]) es un buen resumen, pero puede esconder que una clase tenga un **recall** claramente menor que las otras: si trompeta está casi perfecta, eleva el promedio aunque violín tenga varios errores. Por eso hay que mirar precision y recall por clase. Además, con unas 20 imágenes de test por clase, cada error mueve la métrica de esa clase en unos 5 puntos, así que las diferencias pequeñas no son concluyentes.
4. **¿Qué cambiaría primero en el dataset?** Revisaría las imágenes mal clasificadas para ver si son errores de etiqueta (violas o chelos dentro de violín, guitarras eléctricas dentro de guitarra) y las limpiaría. Después agregaría más ejemplos de los casos difíciles: violines y guitarras fotografiados en el mismo tipo de escena, con fondos variados, y no solo fotos de catálogo con fondo blanco, para que el modelo no use el fondo como atajo.

# 8. Fine-tuning — EN CASA

Ahora permitiremos que una pequeña parte de EfficientNet se adapte al nuevo problema.

> **Concepto clave — Fine-tuning**
>
> En la fase anterior, los pesos del backbone no cambiaron. Ahora descongelaremos únicamente una parte final del modelo y la actualizaremos con un **learning rate mucho menor**.

> **Concepto clave — Batch Normalization**
>
> EfficientNet contiene capas `BatchNormalization`, que mantienen estadísticas internas aprendidas durante el entrenamiento. En datasets pequeños conviene mantenerlas congeladas durante fine-tuning para evitar actualizaciones inestables de esas estadísticas.

Después de cambiar qué capas son entrenables, es necesario **recompilar** el modelo.

In [ ]:
base_model.trainable = True

# TODO 5 — Decide cuántas capas finales adaptar.
# Empieza con ~20. Puedes justificar otro valor usando validation.
N_UNFROZEN = 20

for layer in base_model.layers[:-N_UNFROZEN]:
    layer.trainable = False

# Mantener Batch Normalization congelado.
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

print(
    "Capas entrenables de la base:",
    sum(layer.trainable for layer in base_model.layers),
    "/",
    len(base_model.layers),
)

## TODO 5 — EN CASA: realiza fine-tuning

Recompila y continúa el entrenamiento.

Usa como punto de partida:

- `Adam(learning_rate=1e-5)`;
- `SparseCategoricalCrossentropy`;
- `accuracy`;
- entre **3 y 8 épocas** adicionales;
- `EarlyStopping` sobre `val_loss`.

Tu decisión principal aquí es **cuántas capas descongelar**. Justifícala usando el desempeño de validación, no el conjunto de prueba.

**Decisión:** se descongelan las **últimas 20 capas** de EfficientNetB0, que corresponden aproximadamente al último bloque MBConv (`block7a`) y a la convolución final `top_conv` (la celda anterior imprime cuántas quedan entrenables). Las capas finales son las que codifican rasgos más específicos de ImageNet (partes de objetos), mientras que las primeras capas detectan bordes y texturas genéricas que sirven igual para nuestro problema. Con unas 100 imágenes por clase, descongelar muchas capas daría demasiados parámetros nuevos y riesgo de overfitting.

El *learning rate* se baja a $10^{-5}$, cien veces menor que en la fase anterior, para hacer ajustes pequeños sin destruir lo que la red ya sabe. Se usan 8 épocas como máximo con `EarlyStopping`, que detiene el entrenamiento si `val_loss` no mejora en 2 épocas y restaura los mejores pesos.

Si se quiere justificar otro valor, la forma correcta es repetir esta fase con, por ejemplo, 10, 20 y 40 capas (partiendo cada vez de los mismos pesos de la fase congelada) y quedarse con el que dé menor `val_loss`, sin mirar test.

In [ ]:
# TODO 5 — EN CASA: completa el fine-tuning

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

early_stopping_ft = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
)

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,  # entre 3 y 8
    callbacks=[early_stopping_ft],
)

In [ ]:
plt.figure(figsize=(12, 4))
for k, metric in enumerate(["accuracy", "loss"]):
    plt.subplot(1, 2, k + 1)
    full_train = history.history[metric] + history_ft.history[metric]
    full_val = history.history[f"val_{metric}"] + history_ft.history[f"val_{metric}"]
    plt.plot(full_train, label=f"train {metric}")
    plt.plot(full_val, label=f"val {metric}")
    plt.axvline(len(history.history[metric]) - 0.5, color="gray", ls="--", label="inicio fine-tuning")
    plt.xlabel("Epoch")
    plt.ylabel(metric)
    plt.legend()
plt.tight_layout()
plt.show()

print(f"Mejor val_loss congelado   : {min(history.history['val_loss']):.4f}")
print(f"Mejor val_loss fine-tuning : {min(history_ft.history['val_loss']):.4f}")

# 9. Comparación final — EN CASA

Compara el desempeño del mismo sistema **antes y después** del fine-tuning.

Una mejora pequeña también puede ser válida. Si el resultado empeora, no significa que el experimento “salió mal”: debes interpretar por qué pudo ocurrir.

In [ ]:
test_loss_after, test_acc_after = model.evaluate(test_ds, verbose=0)
print(f"Antes del fine-tuning  : {test_acc_before:.4f}")
print(f"Después del fine-tuning: {test_acc_after:.4f}")
print(f"Diferencia              : {test_acc_after-test_acc_before:+.4f}")

In [ ]:
y_true_ft, y_pred_ft, probabilities_ft = get_predictions(model, test_ds)
print(classification_report(y_true_ft, y_pred_ft, target_names=class_names, digits=3))
cm_ft = confusion_matrix(y_true_ft, y_pred_ft)
ConfusionMatrixDisplay(confusion_matrix=cm_ft, display_labels=class_names).plot(cmap="Blues", xticks_rotation=45)
plt.title("Matriz de confusión — después del fine-tuning")
plt.show()

### Comparación por clase

Tabla de apoyo con la métrica F1 por clase antes y después del fine-tuning.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score
f1_before = f1_score(y_true, y_pred, average=None)
f1_after = f1_score(y_true_ft, y_pred_ft, average=None)
print(f"{'clase':10s} {'F1 antes':>9s} {'F1 después':>11s} {'cambio':>8s}")
for name, a, b in zip(class_names, f1_before, f1_after):
    print(f"{name:10s} {a:9.3f} {b:11.3f} {b - a:+8.3f}")
print(f"\nErrores en test antes  : {int(np.sum(y_true != y_pred))} de {len(y_true)}")
print(f"Errores en test después: {int(np.sum(y_true_ft != y_pred_ft))} de {len(y_true_ft)}")

# 10. Analiza los errores — EN CASA

Una métrica no explica por qué falla un modelo. Debes inspeccionar ejemplos clasificados incorrectamente.

In [ ]:
def show_mistakes(dataset, model, class_names, max_images=12):
    mistakes = []
    for images, labels in dataset:
        probs = model.predict(images, verbose=0)
        preds = probs.argmax(axis=1)
        for image, true_label, pred_label, prob in zip(images.numpy(), labels.numpy(), preds, probs):
            if true_label != pred_label:
                mistakes.append((image.astype("uint8"), int(true_label), int(pred_label), float(prob[pred_label])))
    random.shuffle(mistakes)
    mistakes = mistakes[:max_images]
    if not mistakes:
        print("No se encontraron errores en este conjunto.")
        return
    cols = 4
    rows = int(np.ceil(len(mistakes)/cols))
    plt.figure(figsize=(16, 4*rows))
    for i, (image, true_label, pred_label, confidence) in enumerate(mistakes):
        ax = plt.subplot(rows, cols, i+1)
        ax.imshow(image)
        ax.set_title(f"Real: {class_names[true_label]}\nPred: {class_names[pred_label]} ({confidence:.2f})")
        ax.axis("off")
    plt.tight_layout()

show_mistakes(test_ds, model, class_names)

Para llenar la tabla con precisión, esta celda lista **todos** los errores con el nombre de archivo, de modo que se puedan abrir y revisar uno por uno.

In [ ]:
test_paths = keras.utils.image_dataset_from_directory(
    SPLIT_DIR / "test", image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="int", shuffle=False, verbose=False).file_paths

wrong = np.flatnonzero(y_true_ft != y_pred_ft)
print(f"{len(wrong)} errores en test\n")
for k, i in enumerate(wrong, 1):
    print(f"{k:2d}. {Path(test_paths[i]).parent.name}/{Path(test_paths[i]).name:10s} "
          f"real = {class_names[y_true_ft[i]]:9s} pred = {class_names[y_pred_ft[i]]:9s} "
          f"confianza = {probabilities_ft[i].max():.2f}")

k = min(len(wrong), 8)
if k:
    plt.figure(figsize=(3 * k, 3.4))
    for j, i in enumerate(wrong[:k]):
        ax = plt.subplot(1, k, j + 1)
        with Image.open(test_paths[i]) as img:
            ax.imshow(img)
        ax.set_title(f"{j + 1}. real {class_names[y_true_ft[i]]}\npred {class_names[y_pred_ft[i]]}", fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## TODO 6 — EN CASA: análisis de errores y conclusiones

Selecciona al menos **5 errores** del modelo y analiza su causa probable.

| Imagen/error | Clase real | Predicción | Causa probable | ¿Cómo lo mejorarías? |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |

*(La tabla se llena con los errores listados en la celda anterior. Como guía, estos son los tipos de error que se esperan en este problema y cómo se analizarían:)*

| Tipo de error esperado | Clase real | Predicción | Causa probable | ¿Cómo lo mejorarías? |
|---|---|---|---|---|
| Foto de una viola o un chelo en la carpeta de violín | violin | guitarra | Etiqueta incorrecta: no es un violín | Eliminarla y agregar "violin" con términos más precisos |
| Violín o guitarra fotografiados de lado o de espaldas | violin / guitarra | la otra | Clase visualmente similar: no se ven la boca, los agujeros en *f* ni la voluta | Más ejemplos con vistas variadas |
| Persona tocando en un concierto, instrumento pequeño | cualquiera | cualquiera | Objeto pequeño y oclusión por manos y cuerpo | Recortar el instrumento o incluir más fotos de este tipo en train |
| Producto sobre fondo blanco con marca de agua | guitarra | violin | Sesgo del dataset: el modelo asoció el fondo con una clase | Balancear el tipo de fondo entre clases |
| Trompeta junto a otros instrumentos o dibujo | trompeta | otra | Imagen ambigua o que no es fotografía | Limpiar el dataset; la etiqueta de una imagen con varios objetos es ambigua |

Posibles causas: imagen ambigua, etiqueta incorrecta, objeto pequeño, oclusión, fondo engañoso, clase visualmente similar, mala calidad, sesgo del dataset o error razonable del modelo.

Después, escribe una conclusión breve respondiendo:

1. ¿Qué aprendiste al construir el dataset?
2. ¿Cuál fue el principal problema de calidad?
3. ¿Cuánto cambió el desempeño después del fine-tuning?
4. ¿El fine-tuning ayudó? ¿Por qué crees que sí o no?
5. ¿Qué mejorarías primero: datos, entrenamiento o arquitectura?

**Conclusiones** *(los valores entre corchetes se completan con los resultados de la ejecución)*

1. **¿Qué aprendí al construir el dataset?** Que la parte más difícil y la que más influye en el resultado no es el modelo sino los datos. Un buscador devuelve muchas imágenes que no son de la clase (instrumentos parecidos, dibujos, logos) y la calidad depende mucho de cómo se formulen los términos de búsqueda: agregar *instrument photo* o *close up* ayudó más que pedir más resultados. También que la validación técnica (que el archivo abra) y la semántica (que la imagen sea correcta) son problemas distintos, y que los casi duplicados son una fuente silenciosa de *data leakage*.
2. **Principal problema de calidad:** [completar; se espera la mezcla con instrumentos parecidos, violas y chelos en violín y guitarras eléctricas en guitarra, y las fotos de catálogo con fondo blanco].
3. **Cambio después del fine-tuning:** el accuracy en test pasó de [ ] a [ ] (diferencia [ ]), es decir, [ ] errores menos de [ ] imágenes.
4. **¿Ayudó el fine-tuning?** Se espera una mejora pequeña o nula. La base congelada ya parte de características muy buenas, porque ImageNet incluye clases como *acoustic guitar*, *violin* y *cornet*, así que el primer modelo probablemente ya queda por encima de 90 %. Con poco margen y un test de unas 60 imágenes, una diferencia de una o dos imágenes no es estadísticamente significativa. Si mejora, es porque las últimas capas se adaptan a los detalles que separan guitarra de violín; si empeora, lo más probable es overfitting a los pocos ejemplos de entrenamiento o simple ruido del test pequeño.
5. **¿Qué mejoraría primero?** **Los datos.** Los errores observados vienen principalmente de etiquetas incorrectas, imágenes ambiguas y sesgos de fondo; ningún cambio de arquitectura corrige una etiqueta mala. Primero limpiaría y ampliaría el dataset (más imágenes, fondos variados, casos difíciles entre guitarra y violín), luego ajustaría el entrenamiento (cuántas capas descongelar, elegido con validation) y solo al final probaría otro backbone.

# Extensión opcional — Prueba con una imagen externa

Busca una imagen que **no esté en tu dataset** y prueba el modelo. Esta parte no cuenta como un TODO principal, pero sirve como comprobación cualitativa de generalización.

In [ ]:
# OPCIONAL
# EXTERNAL_IMAGE_URL = "https://..."
# response = requests.get(EXTERNAL_IMAGE_URL, timeout=10)
# response.raise_for_status()
# img = Image.open(io.BytesIO(response.content)).convert("RGB")
# img_resized = img.resize(IMG_SIZE)
# x = np.array(img_resized, dtype="float32")[None, ...]
# p = model.predict(x, verbose=0)[0]
# plt.imshow(img)
# plt.axis("off")
# plt.title(f"{class_names[p.argmax()]} — confianza {p.max():.2f}")
# plt.show()

# 🏠 Cierre conceptual — ENTREGA

Como parte del **TODO 6**, explica con tus propias palabras la diferencia entre:

- **feature extraction / transfer learning con base congelada**;
- **fine-tuning**;
- **entrenar una CNN desde cero**.

No describas solo el código. Explica **qué parámetros se están aprendiendo y de dónde viene el conocimiento inicial**.

**Respuesta:**

**Transfer learning con base congelada (feature extraction).** Los pesos de EfficientNetB0 se toman tal cual vienen de ImageNet y **no se modifican**: la red base funciona como un extractor fijo que convierte cada imagen en un vector de 1280 características. Lo único que se aprende es la nueva cabeza: la capa `Dense(3)` con $1280 \times 3 + 3 = 3843$ parámetros. Todo el conocimiento visual (bordes, texturas, formas, partes de objetos) viene del entrenamiento previo con 1.2 millones de imágenes de ImageNet; nuestro dataset solo enseña cómo combinar esas características para separar nuestras tres clases. Es rápido y funciona bien con pocos datos.

**Fine-tuning.** Se parte del modelo anterior y se **descongelan las últimas capas** de la base para que sus pesos también se actualicen, con un *learning rate* muy pequeño. Ahora se aprenden la cabeza y los parámetros de esas capas finales. El conocimiento inicial sigue viniendo de ImageNet, pero las características de alto nivel se ajustan ligeramente a nuestro problema (por ejemplo, a los detalles que distinguen un violín de una guitarra). Las capas iniciales siguen congeladas porque sus rasgos genéricos ya sirven, y las capas de Batch Normalization también, para no alterar sus estadísticas con lotes pequeños.

**Entrenar una CNN desde cero.** Todos los pesos se inicializan al azar y **todos** se aprenden con nuestros datos. No hay conocimiento previo: la red tendría que descubrir por sí misma qué es un borde, una textura y una forma usando unas 300 imágenes. EfficientNetB0 tiene unos 4 millones de parámetros, así que con tan pocos datos memorizaría el conjunto de entrenamiento y generalizaría mal. Entrenar desde cero solo tiene sentido con datasets grandes o con imágenes muy distintas de las de ImageNet (por ejemplo, imágenes médicas o satelitales).

# Reto opcional

Elige **uno**:

**A. Más datos:** duplica el número de imágenes manteniendo el modelo igual.  
**B. Otro backbone:** prueba MobileNetV3, ResNet50 o EfficientNetV2B0.  
**C. Dataset difícil:** agrega una cuarta clase visualmente parecida.  
**D. Ablation:** entrena sin `data_augmentation` y compara.

Cambia **una sola variable a la vez**.

# ✅ Checklist de entrega

Antes de entregar verifica:

### Evidencia del trabajo en clase
- [ ] Problema y clases definidos.
- [ ] Dataset construido y revisado.
- [ ] Train / validation / test creados.
- [ ] Primer modelo con backbone congelado entrenado.

### Trabajo en casa
- [ ] Evaluación sobre test.
- [ ] Matriz de confusión.
- [ ] Precision / recall / F1 interpretados.
- [ ] Fine-tuning ejecutado.
- [ ] Comparación antes vs. después del fine-tuning.
- [ ] Al menos 5 errores analizados.
- [ ] Conclusión final escrita.

### Opcional
- [ ] Imagen externa.
- [ ] Experimento adicional / reto.

# Rúbrica sugerida

| Componente | Peso |
|---|---:|
| TODO 1 — Problema y estrategia de búsqueda | 10% |
| TODO 2 — Construcción, inspección y limpieza del dataset | 20% |
| Split y pipeline de datos | 10% |
| TODO 3 — Transfer learning + curvas | 20% |
| TODO 4 — Evaluación e interpretación | 10% |
| TODO 5 — Fine-tuning y comparación | 15% |
| TODO 6 — Análisis de errores + conclusión conceptual | 15% |
| **Total** | **100%** |